# Hearo — YAMNet 한국 가정환경음 분류 v2

이 노트북은 기존 yamnet_fine_tuning.ipynb를 보존하면서 다음을 자동 수행합니다.

- metadata.csv 기반 source/session 그룹 분할
- 9개 표적음 + 비표적음 학습 및 오알림 평가
- YAMNet 프레임 임베딩 기반 최대 3라운드 모델 선택
- untouched test 1회 평가, calibration, threshold 선택
- Keras/TFLite 일치 검증 및 Raspberry Pi용 메타데이터 내보내기

실제 음원은 Google Drive에서 읽습니다. 이 파일 자체에는 실행 결과를 미리 기록하지 않습니다.



## 1. Colab 환경 설정



In [ ]:
!pip -q install tensorflow-hub resampy soundfile pydub seaborn scikit-learn
!apt-get -qq update
!apt-get -qq install -y ffmpeg fonts-nanum



In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import time
import unicodedata
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import resampy
import seaborn as sns
import soundfile as sf
import tensorflow as tf
import tensorflow_hub as hub
from google.colab import drive
from IPython.display import display
from pydub import AudioSegment
from scipy.optimize import minimize_scalar
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

SEED = 42
TARGET_SR = 16_000
YAMNET_FRAME_SECONDS = 0.96
MIN_SAMPLES = int(TARGET_SR * YAMNET_FRAME_SECONDS)
YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"
UNKNOWN_LABEL = "비표적음"
TARGET_CATEGORIES = [
    "노크_목재",
    "노크_철재문",
    "도어락_개방음",
    "도어락_입력음",
    "사이렌_삐뽀삐뽀",
    "사이렌_안내음",
    "사이렌_애애애애앵",
    "사이렌_철철철",
    "아기 울음",
]
CATEGORIES = TARGET_CATEGORIES + [UNKNOWN_LABEL]
NUM_CLASSES = len(CATEGORIES)
UNKNOWN_ID = CATEGORIES.index(UNKNOWN_LABEL)
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a"}

DATA_DIR = Path("/content/drive/MyDrive/소리 정리")
METADATA_PATH = DATA_DIR / "metadata.csv"
OUTPUT_DIR = Path("/content/drive/MyDrive/Hearo_model_v2")
CACHE_DIR = OUTPUT_DIR / "embedding_cache"
FIG_DIR = OUTPUT_DIR / "figures"

MAX_EPOCHS = 80
BATCH_SIZE = 64
AUGMENTATIONS_PER_FILE = 2
MAX_FALSE_ALERT_RATE = 0.05
MIN_PROMOTION = 0.005
BOOTSTRAP_ITERATIONS = 2_000

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

drive.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("출력 경로:", OUTPUT_DIR)



## 2. 오디오 로드와 metadata.csv 검증

metadata.csv 필수 열은 relative_path, label, group_id입니다. relative_path는 DATA_DIR 기준 경로이며,
같은 녹음 세션 또는 같은 원본에서 파생된 파일은 반드시 같은 group_id를 사용해야 합니다.



In [ ]:
def nfc(value: Any) -> str:
    return unicodedata.normalize("NFC", str(value).strip())


def load_audio(path: str | Path, pad_short: bool = True) -> np.ndarray:
    """지원 음원을 mono 16 kHz float32 [-1, 1]로 변환합니다."""
    path = Path(path)
    ext = path.suffix.lower()
    if ext not in AUDIO_EXTENSIONS:
        raise ValueError(f"지원하지 않는 확장자: {path}")

    if ext in {".mp3", ".m4a"}:
        segment = AudioSegment.from_file(path)
        segment = segment.set_channels(1).set_frame_rate(TARGET_SR)
        raw = np.asarray(segment.get_array_of_samples())
        scale = float(1 << (8 * segment.sample_width - 1))
        waveform = raw.astype(np.float32) / scale
    else:
        waveform, sample_rate = sf.read(path, dtype="float32", always_2d=False)
        if waveform.ndim == 2:
            waveform = waveform.mean(axis=1)
        if int(sample_rate) != TARGET_SR:
            waveform = resampy.resample(waveform, int(sample_rate), TARGET_SR)

    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)
    if waveform.size == 0 or not np.all(np.isfinite(waveform)):
        raise ValueError(f"비어 있거나 유효하지 않은 음원: {path}")
    waveform = np.clip(waveform, -1.0, 1.0)
    if pad_short and len(waveform) < MIN_SAMPLES:
        waveform = np.pad(waveform, (0, MIN_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)


def decoded_audio_hash(waveform: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(waveform, np.float32).tobytes()).hexdigest()


def build_normalized_file_index(root: Path) -> dict[str, Path]:
    index: dict[str, Path] = {}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            rel = nfc(path.relative_to(root).as_posix())
            if rel in index:
                raise ValueError(f"NFC 정규화 후 경로가 충돌합니다: {rel}")
            index[rel] = path
    return index


def validate_and_load_metadata(metadata_path: Path, data_dir: Path) -> pd.DataFrame:
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"{metadata_path}가 없습니다. relative_path,label,group_id 열을 만들어 주세요."
        )
    raw = pd.read_csv(metadata_path, dtype=str, keep_default_na=False)
    required = {"relative_path", "label", "group_id"}
    missing_columns = required - set(raw.columns)
    if missing_columns:
        raise ValueError(f"metadata.csv 필수 열 누락: {sorted(missing_columns)}")

    df = raw[["relative_path", "label", "group_id"]].copy()
    for column in required:
        df[column] = df[column].map(nfc)
    if (df[list(required)] == "").any().any():
        raise ValueError("metadata.csv에 빈 relative_path, label 또는 group_id가 있습니다.")
    if df["relative_path"].duplicated().any():
        duplicates = df.loc[df["relative_path"].duplicated(False), "relative_path"].tolist()
        raise ValueError(f"metadata.csv 중복 경로: {duplicates[:10]}")

    unexpected_labels = sorted(set(df["label"]) - set(CATEGORIES))
    missing_labels = sorted(set(CATEGORIES) - set(df["label"]))
    if unexpected_labels or missing_labels:
        raise ValueError(
            f"레이블 불일치. 예상 밖={unexpected_labels}, 데이터에 없음={missing_labels}"
        )

    file_index = build_normalized_file_index(data_dir)
    listed = set(df["relative_path"])
    missing_files = sorted(listed - set(file_index))
    unlisted_files = sorted(set(file_index) - listed)
    if missing_files or unlisted_files:
        raise ValueError(
            "metadata.csv와 실제 파일이 일치하지 않습니다. "
            f"없는 파일={missing_files[:10]}, 미등록 파일={unlisted_files[:10]}"
        )

    records = []
    hash_to_label: dict[str, str] = {}
    errors = []
    for row in df.itertuples(index=False):
        path = file_index[row.relative_path]
        try:
            waveform = load_audio(path)
            audio_hash = decoded_audio_hash(waveform)
            previous_label = hash_to_label.get(audio_hash)
            if previous_label is not None and previous_label != row.label:
                raise ValueError(f"동일 음원이 서로 다른 레이블에 존재: {previous_label}, {row.label}")
            hash_to_label[audio_hash] = row.label
            records.append({
                "relative_path": row.relative_path,
                "filepath": str(path),
                "label": row.label,
                "label_id": CATEGORIES.index(row.label),
                "group_id": row.group_id,
                "audio_hash": audio_hash,
                "duration_seconds": len(waveform) / TARGET_SR,
                "rms": float(np.sqrt(np.mean(np.square(waveform)) + 1e-12)),
                "peak": float(np.max(np.abs(waveform))),
            })
        except Exception as exc:
            errors.append(f"{row.relative_path}: {exc}")
    if errors:
        raise ValueError("음원 검증 실패:\n" + "\n".join(errors[:30]))

    validated = pd.DataFrame(records).sort_values("relative_path").reset_index(drop=True)
    duplicate_hashes = validated[validated["audio_hash"].duplicated(False)]
    if not duplicate_hashes.empty:
        print("경고: 동일 레이블 내 완전 중복 음원이 있습니다.")
        display(duplicate_hashes[["relative_path", "label", "group_id", "audio_hash"]])
    return validated


metadata = validate_and_load_metadata(METADATA_PATH, DATA_DIR)
display(metadata.groupby("label").agg(files=("relative_path", "size"), groups=("group_id", "nunique"), seconds=("duration_seconds", "sum")))
metadata.to_csv(OUTPUT_DIR / "validated_manifest.csv", index=False, encoding="utf-8-sig")



## 3. 누수 없는 outer test / inner CV 분할



In [ ]:
REQUIRED_CLASS_IDS = set(range(NUM_CLASSES))


def missing_classes(df: pd.DataFrame, indices: np.ndarray) -> set[int]:
    return REQUIRED_CLASS_IDS - set(df.iloc[indices]["label_id"].astype(int))


def groups_per_class(df: pd.DataFrame, indices: np.ndarray | None = None) -> pd.Series:
    subset = df if indices is None else df.iloc[indices]
    return (
        subset.groupby("label_id")["group_id"].nunique()
        .reindex(range(NUM_CLASSES), fill_value=0)
        .astype(int)
    )


def feasible_splits(df: pd.DataFrame, requested: int, name: str) -> int:
    counts = groups_per_class(df)
    count = int(min(requested, counts.min()))
    if count < 3:
        readable = pd.Series(counts.to_numpy(), index=CATEGORIES)
        raise ValueError(
            f"{name} grouped split에는 클래스마다 최소 3개 group_id가 필요합니다.\n{readable}"
        )
    return count


def distribution_distance(labels: np.ndarray, indices: np.ndarray) -> float:
    full = np.bincount(labels, minlength=NUM_CLASSES) / len(labels)
    part = np.bincount(labels[indices], minlength=NUM_CLASSES) / len(indices)
    return float(np.abs(full - part).sum())


def holdout_score(df: pd.DataFrame, test_idx: np.ndarray, test_size: float) -> float:
    labels = df["label_id"].to_numpy(dtype=np.int64)
    full_counts = np.bincount(labels, minlength=NUM_CLASSES)
    test_counts = np.bincount(labels[test_idx], minlength=NUM_CLASSES)
    per_class_fraction = test_counts / np.maximum(full_counts, 1)
    class_ratio_error = float(np.mean(np.abs(per_class_fraction - test_size)))
    row_ratio_error = abs(len(test_idx) / len(df) - test_size)
    group_ratio = df.iloc[test_idx]["group_id"].nunique() / df["group_id"].nunique()
    group_ratio_error = abs(group_ratio - test_size)
    return class_ratio_error + 0.25 * row_ratio_error + 0.05 * group_ratio_error


def select_outer_holdout(
    df: pd.DataFrame, test_size: float = 0.20, trials: int = 512, seed: int = SEED
) -> tuple[np.ndarray, np.ndarray, float]:
    all_group_counts = groups_per_class(df)
    if (all_group_counts < 4).any():
        readable = pd.Series(all_group_counts.to_numpy(), index=CATEGORIES)
        raise ValueError(
            "최종 test와 최소 3-fold CV를 함께 만들려면 클래스마다 최소 4개 group_id가 필요합니다."
            f"\n{readable}"
        )

    splitter = GroupShuffleSplit(n_splits=trials, test_size=test_size, random_state=seed)
    best: tuple[float, np.ndarray, np.ndarray] | None = None
    for train_idx, test_idx in splitter.split(df, df["label_id"], df["group_id"]):
        if missing_classes(df, train_idx) or missing_classes(df, test_idx):
            continue
        # test 격리 뒤 development에 클래스별 group_id를 3개 이상 남겨야 합니다.
        if (groups_per_class(df, train_idx) < 3).any():
            continue
        score = holdout_score(df, test_idx, test_size)
        if best is None or score < best[0] - 1e-12:
            best = (score, train_idx.copy(), test_idx.copy())

    if best is None:
        raise ValueError(
            "모든 클래스를 포함하면서 development에 클래스별 group_id 3개를 남기는 "
            "outer test split을 찾지 못했습니다. 희소 클래스의 독립 원본을 추가하세요."
        )
    return best[1], best[2], best[0]


def select_inner_splits(
    df: pd.DataFrame, n_splits: int, trials: int = 64, seed: int = SEED + 1
) -> tuple[list[tuple[np.ndarray, np.ndarray]], float, int]:
    labels = df["label_id"].to_numpy(dtype=np.int64)
    best: tuple[float, list[tuple[np.ndarray, np.ndarray]], int] | None = None
    for offset in range(trials):
        candidate_seed = seed + offset
        splitter = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True, random_state=candidate_seed
        )
        candidate = list(splitter.split(df, labels, df["group_id"]))
        if any(
            missing_classes(df, train_idx) or missing_classes(df, val_idx)
            for train_idx, val_idx in candidate
        ):
            continue
        fold_sizes = np.asarray([len(val_idx) for _, val_idx in candidate], dtype=np.float64)
        score = float(np.mean([distribution_distance(labels, val_idx) for _, val_idx in candidate]))
        score += float(np.std(fold_sizes) / len(df))
        if best is None or score < best[0] - 1e-12:
            best = (score, [(a.copy(), b.copy()) for a, b in candidate], candidate_seed)

    if best is None:
        raise ValueError(
            f"모든 클래스가 train/validation에 존재하는 {n_splits}-fold grouped CV를 "
            "찾지 못했습니다. 희소 클래스의 독립 원본을 추가하세요."
        )
    return best[1], best[0], best[2]


outer_train_idx, test_idx, outer_score = select_outer_holdout(metadata)
# 같은 seed에서 선택 결과가 완전히 재현되는지 즉시 확인합니다.
outer_train_repeat, test_repeat, outer_score_repeat = select_outer_holdout(metadata)
assert np.array_equal(outer_train_idx, outer_train_repeat)
assert np.array_equal(test_idx, test_repeat)
assert np.isclose(outer_score, outer_score_repeat)

dev_df = metadata.iloc[outer_train_idx].reset_index(drop=True)
test_df = metadata.iloc[test_idx].reset_index(drop=True)
assert set(dev_df["group_id"]).isdisjoint(set(test_df["group_id"]))

for split_name, split_df in [("development", dev_df), ("test", test_df)]:
    missing = REQUIRED_CLASS_IDS - set(split_df["label_id"].astype(int))
    if missing:
        raise AssertionError(f"{split_name} split에 빠진 클래스: {[CATEGORIES[i] for i in sorted(missing)]}")

inner_splits_count = feasible_splits(dev_df, 4, "inner")
inner_splits, inner_score, inner_seed = select_inner_splits(dev_df, inner_splits_count)
inner_repeat, inner_score_repeat, inner_seed_repeat = select_inner_splits(dev_df, inner_splits_count)
assert inner_seed == inner_seed_repeat and np.isclose(inner_score, inner_score_repeat)
assert all(
    np.array_equal(a_train, b_train) and np.array_equal(a_val, b_val)
    for (a_train, a_val), (b_train, b_val) in zip(inner_splits, inner_repeat)
)
for train_idx, val_idx in inner_splits:
    assert set(dev_df.iloc[train_idx]["group_id"]).isdisjoint(set(dev_df.iloc[val_idx]["group_id"]))
    assert not missing_classes(dev_df, train_idx)
    assert not missing_classes(dev_df, val_idx)

split_manifest = metadata[["relative_path", "label", "group_id"]].copy()
split_manifest["split"] = np.where(split_manifest["relative_path"].isin(test_df["relative_path"]), "test", "development")
split_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False, encoding="utf-8-sig")
print(
    f"Development={len(dev_df)}, Test={len(test_df)} "
    f"({len(test_df) / len(metadata):.1%}), inner folds={inner_splits_count}, inner seed={inner_seed}"
)
display(pd.DataFrame({
    "development_files": dev_df.groupby("label").size().reindex(CATEGORIES),
    "test_files": test_df.groupby("label").size().reindex(CATEGORIES),
    "development_groups": dev_df.groupby("label")["group_id"].nunique().reindex(CATEGORIES),
    "test_groups": test_df.groupby("label")["group_id"].nunique().reindex(CATEGORIES),
}))



## 4. YAMNet 프레임 특징과 안전한 증강 캐시



In [ ]:
print("YAMNet 로딩 중...")
yamnet_model = hub.load(YAMNET_HANDLE)
print("YAMNet 로드 완료")


def stable_seed(*parts: Any) -> int:
    digest = hashlib.sha256("|".join(map(str, parts)).encode("utf-8")).hexdigest()
    return int(digest[:8], 16)


def zero_filled_shift(waveform: np.ndarray, shift_samples: int) -> np.ndarray:
    shifted = np.zeros_like(waveform)
    if shift_samples > 0:
        shifted[shift_samples:] = waveform[:-shift_samples]
    elif shift_samples < 0:
        shifted[:shift_samples] = waveform[-shift_samples:]
    else:
        shifted[:] = waveform
    return shifted


def fit_noise(noise: np.ndarray, length: int, rng: np.random.Generator) -> np.ndarray:
    if len(noise) < length:
        repeats = int(np.ceil(length / len(noise)))
        noise = np.tile(noise, repeats)
    start = int(rng.integers(0, max(1, len(noise) - length + 1)))
    return noise[start:start + length]


def augment_waveform(
    waveform: np.ndarray,
    seed: int,
    noise_paths: list[str],
    allow_noise_mix: bool,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    augmented = waveform.astype(np.float32).copy()

    gain_db = float(rng.uniform(-6.0, 6.0))
    augmented *= np.float32(10.0 ** (gain_db / 20.0))

    max_shift = int(0.1 * TARGET_SR)
    shift = int(rng.integers(-max_shift, max_shift + 1))
    augmented = zero_filled_shift(augmented, shift)

    speed = float(rng.uniform(0.95, 1.05))
    stretched_sr = max(1, int(round(TARGET_SR / speed)))
    augmented = resampy.resample(augmented, TARGET_SR, stretched_sr).astype(np.float32)

    if allow_noise_mix and noise_paths and rng.random() < 0.75:
        noise_path = noise_paths[int(rng.integers(0, len(noise_paths)))]
        noise = fit_noise(load_audio(noise_path), len(augmented), rng)
        signal_rms = float(np.sqrt(np.mean(augmented ** 2) + 1e-12))
        noise_rms = float(np.sqrt(np.mean(noise ** 2) + 1e-12))
        snr_db = float(rng.uniform(10.0, 30.0))
        noise_gain = signal_rms / (max(noise_rms, 1e-8) * 10.0 ** (snr_db / 20.0))
        augmented = augmented + noise * np.float32(noise_gain)

    if len(augmented) < MIN_SAMPLES:
        augmented = np.pad(augmented, (0, MIN_SAMPLES - len(augmented)))
    return np.clip(augmented, -1.0, 1.0).astype(np.float32)


def yamnet_features(waveform: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    scores, embeddings, _ = yamnet_model(tf.convert_to_tensor(waveform, tf.float32))
    return embeddings.numpy().astype(np.float32), scores.numpy().astype(np.float32)


def cached_features(
    row: Any,
    augmentation_index: int = 0,
    noise_paths: list[str] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    waveform = load_audio(row.filepath)
    profile = "clean"
    if augmentation_index > 0:
        profile = f"light_{augmentation_index}"
        waveform = augment_waveform(
            waveform,
            stable_seed(SEED, row.audio_hash, augmentation_index),
            noise_paths or [],
            allow_noise_mix=(row.label != UNKNOWN_LABEL),
        )
    waveform_hash = decoded_audio_hash(waveform)
    cache_key = hashlib.sha256(
        f"{YAMNET_HANDLE}|{waveform_hash}|{profile}".encode("utf-8")
    ).hexdigest()
    cache_path = CACHE_DIR / f"{cache_key}.npz"
    if cache_path.exists():
        try:
            with np.load(cache_path) as cached:
                return cached["embeddings"], cached["scores"]
        except Exception:
            cache_path.unlink(missing_ok=True)
    embeddings, scores = yamnet_features(waveform)
    temporary_path = cache_path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary_path, embeddings=embeddings, scores=scores)
    os.replace(temporary_path, cache_path)
    return embeddings, scores


# clean 특징은 split과 무관한 frozen YAMNet 변환이므로 한 번만 캐시합니다.
for index, row in enumerate(metadata.itertuples(index=False), start=1):
    cached_features(row)
    if index % 25 == 0 or index == len(metadata):
        print(f"Clean embedding cache: {index}/{len(metadata)}")



## 5. 분류기, 프레임 집계 및 운영 지표



In [ ]:
@dataclass(frozen=True)
class FamilyConfig:
    name: str
    head: str
    representation: str
    augmentation: str = "none"
    class_weighting: bool = False


def balanced_class_weights(rows: pd.DataFrame) -> dict[int, float]:
    counts = rows.groupby("label_id").size().reindex(range(NUM_CLASSES), fill_value=0)
    if (counts == 0).any():
        raise ValueError(f"학습 fold에 빠진 클래스가 있습니다: {counts.to_dict()}")
    return {int(i): float(len(rows) / (NUM_CLASSES * count)) for i, count in counts.items()}


def build_training_arrays(
    rows: pd.DataFrame,
    config: FamilyConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    noise_paths = rows.loc[rows["label"] == UNKNOWN_LABEL, "filepath"].tolist()
    class_weights = balanced_class_weights(rows) if config.class_weighting else {
        i: 1.0 for i in range(NUM_CLASSES)
    }
    variants = 1 + (AUGMENTATIONS_PER_FILE if config.augmentation == "light" else 0)
    features: list[np.ndarray] = []
    labels: list[np.ndarray] = []
    weights: list[np.ndarray] = []

    for row in rows.itertuples(index=False):
        for augmentation_index in range(variants):
            embeddings, _ = cached_features(row, augmentation_index, noise_paths)
            if config.representation == "clip_mean":
                current = embeddings.mean(axis=0, keepdims=True)
            else:
                current = embeddings
            count = len(current)
            features.append(current)
            labels.append(np.full(count, row.label_id, dtype=np.int64))
            # 원본 파일별 총 가중치는 같고, class weighting만 추가로 반영됩니다.
            per_sample = class_weights[row.label_id] / (variants * count)
            weights.append(np.full(count, per_sample, dtype=np.float32))

    x = np.concatenate(features).astype(np.float32)
    y = np.concatenate(labels).astype(np.int64)
    sample_weight = np.concatenate(weights).astype(np.float32)
    sample_weight *= len(sample_weight) / sample_weight.sum()
    return x, y, sample_weight


def build_validation_arrays(
    rows: pd.DataFrame,
    representation: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    features: list[np.ndarray] = []
    labels: list[np.ndarray] = []
    weights: list[np.ndarray] = []
    for row in rows.itertuples(index=False):
        embeddings, _ = cached_features(row)
        current = embeddings.mean(axis=0, keepdims=True) if representation == "clip_mean" else embeddings
        features.append(current)
        labels.append(np.full(len(current), row.label_id, dtype=np.int64))
        weights.append(np.full(len(current), 1.0 / len(current), dtype=np.float32))
    x = np.concatenate(features).astype(np.float32)
    y = np.concatenate(labels).astype(np.int64)
    sample_weight = np.concatenate(weights).astype(np.float32)
    sample_weight *= len(sample_weight) / sample_weight.sum()
    return x, y, sample_weight


def build_classifier(
    config: FamilyConfig,
    x_train: np.ndarray,
    seed: int,
    normalization_weights: np.ndarray | None = None,
) -> tf.keras.Model:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    inputs = tf.keras.layers.Input(shape=(1024,), dtype=tf.float32, name="embedding")

    if config.head == "baseline":
        x = tf.keras.layers.Dense(256, activation="relu", name="dense_256")(inputs)
        x = tf.keras.layers.Dropout(0.3, seed=seed, name="dropout_1")(x)
        x = tf.keras.layers.Dense(128, activation="relu", name="dense_128")(x)
        x = tf.keras.layers.Dropout(0.3, seed=seed + 1, name="dropout_2")(x)
    else:
        if normalization_weights is None:
            normalization_weights = np.ones(len(x_train), dtype=np.float32)
        mean = np.average(x_train, axis=0, weights=normalization_weights)
        variance = np.average((x_train - mean) ** 2, axis=0, weights=normalization_weights) + 1e-6
        x = tf.keras.layers.Normalization(
            axis=-1,
            mean=mean,
            variance=variance,
            name="embedding_normalization",
        )(inputs)
        if config.head == "compact":
            x = tf.keras.layers.Dense(
                128,
                activation="swish",
                kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                name="compact_dense",
            )(x)
            x = tf.keras.layers.Dropout(0.25, seed=seed, name="compact_dropout")(x)
        elif config.head != "linear":
            raise ValueError(f"알 수 없는 head: {config.head}")

    output_regularizer = None if config.head == "baseline" else tf.keras.regularizers.l2(1e-4)
    logits = tf.keras.layers.Dense(NUM_CLASSES, kernel_regularizer=output_regularizer, name="logits")(x)
    model = tf.keras.Model(inputs, logits, name=f"hearo_{config.head}")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def softmax_np(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    scaled = np.asarray(logits, np.float64) / max(float(temperature), 1e-4)
    scaled -= scaled.max(axis=-1, keepdims=True)
    exp = np.exp(scaled)
    return (exp / exp.sum(axis=-1, keepdims=True)).astype(np.float64)


def pool_frame_logits(logits: np.ndarray, pooling: str, temperature: float = 1.0) -> np.ndarray:
    logits = np.asarray(logits, np.float64)
    if len(logits) == 1:
        return softmax_np(logits, temperature)[0]
    probabilities = softmax_np(logits, temperature)
    if pooling == "mean_probability":
        pooled = probabilities.mean(axis=0)
    elif pooling == "topk_probability":
        k = max(1, int(math.ceil(len(probabilities) * 0.5)))
        pooled = np.sort(probabilities, axis=0)[-k:].mean(axis=0)
    elif pooling == "logit_logmeanexp":
        # 배포 TFLite는 calibrated probability를 출력하므로 log(probability)에서 집계합니다.
        log_probabilities = np.log(np.clip(probabilities, 1e-12, 1.0))
        maximum = log_probabilities.max(axis=0, keepdims=True)
        pooled_logits = maximum[0] + np.log(
            np.exp(log_probabilities - maximum).mean(axis=0) + 1e-12
        )
        pooled = softmax_np(pooled_logits[None, :])[0]
    else:
        raise ValueError(f"알 수 없는 pooling: {pooling}")
    pooled = np.clip(pooled, 1e-12, None)
    return pooled / pooled.sum()


def aggregate_context(
    frame_logits: np.ndarray,
    pooling: str,
    context_frames: str | int = "full",
    temperature: float = 1.0,
) -> np.ndarray:
    if context_frames == "full":
        return pool_frame_logits(frame_logits, pooling, temperature)
    width = max(1, int(context_frames))
    if len(frame_logits) <= width:
        windows = [frame_logits]
    else:
        windows = [frame_logits[start:start + width] for start in range(len(frame_logits) - width + 1)]
    window_probabilities = np.stack([
        pool_frame_logits(window, pooling, temperature) for window in windows
    ])
    # 실시간 시스템의 "어느 window에서든 알림" 조건을 파일 단위 평가에 반영합니다.
    target_confidence = window_probabilities[:, :UNKNOWN_ID].max(axis=1)
    return window_probabilities[int(np.argmax(target_confidence))]


def decision_predictions(probabilities: np.ndarray, thresholds: float | np.ndarray) -> np.ndarray:
    probabilities = np.asarray(probabilities)
    raw = probabilities.argmax(axis=1)
    confidence = probabilities[np.arange(len(probabilities)), raw]
    threshold_array = np.full(NUM_CLASSES, float(thresholds)) if np.isscalar(thresholds) else np.asarray(thresholds)
    accepted = (raw != UNKNOWN_ID) & (confidence >= threshold_array[raw])
    return np.where(accepted, raw, UNKNOWN_ID).astype(np.int64)


def expected_calibration_error(y_true: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> float:
    confidence = probabilities.max(axis=1)
    prediction = probabilities.argmax(axis=1)
    correct = (prediction == y_true).astype(float)
    error = 0.0
    boundaries = np.linspace(0.0, 1.0, bins + 1)
    for lower, upper in zip(boundaries[:-1], boundaries[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.any():
            error += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(error)


def operating_metrics(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    thresholds: float | np.ndarray,
) -> dict[str, float]:
    prediction = decision_predictions(probabilities, thresholds)
    unknown_mask = y_true == UNKNOWN_ID
    false_alert_rate = float(np.mean(prediction[unknown_mask] != UNKNOWN_ID)) if unknown_mask.any() else float("nan")
    return {
        "target_macro_f1": float(f1_score(y_true, prediction, labels=range(UNKNOWN_ID), average="macro", zero_division=0)),
        "all_class_macro_f1": float(f1_score(y_true, prediction, labels=range(NUM_CLASSES), average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, prediction)),
        "false_alert_rate": false_alert_rate,
        "ece": expected_calibration_error(y_true, probabilities),
    }


def tune_global_threshold(y_true: np.ndarray, probabilities: np.ndarray) -> tuple[float, dict[str, float]]:
    best: tuple[float, float, float, dict[str, float]] | None = None
    for threshold in np.unique(np.concatenate([np.linspace(0.0, 0.995, 200), [1.0]])):
        metrics = operating_metrics(y_true, probabilities, float(threshold))
        feasible = metrics["false_alert_rate"] <= MAX_FALSE_ALERT_RATE + 1e-12
        key = (float(feasible), metrics["target_macro_f1"], metrics["accuracy"])
        if best is None or key > best[:3]:
            best = (*key, {**metrics, "threshold": float(threshold)})
    assert best is not None
    return float(best[3]["threshold"]), best[3]


def tune_class_thresholds(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    initial_threshold: float,
) -> tuple[np.ndarray, dict[str, float]]:
    thresholds = np.full(NUM_CLASSES, initial_threshold, dtype=np.float64)
    thresholds[UNKNOWN_ID] = 0.0
    grid = np.linspace(0.05, 1.0, 96)
    for _ in range(2):
        for class_id in range(UNKNOWN_ID):
            best_threshold = thresholds[class_id]
            best_metrics = operating_metrics(y_true, probabilities, thresholds)
            best_key = (
                float(best_metrics["false_alert_rate"] <= MAX_FALSE_ALERT_RATE + 1e-12),
                best_metrics["target_macro_f1"],
                best_metrics["accuracy"],
            )
            for candidate in grid:
                trial = thresholds.copy()
                trial[class_id] = candidate
                metrics = operating_metrics(y_true, probabilities, trial)
                key = (
                    float(metrics["false_alert_rate"] <= MAX_FALSE_ALERT_RATE + 1e-12),
                    metrics["target_macro_f1"],
                    metrics["accuracy"],
                )
                if key > best_key:
                    best_key, best_threshold, best_metrics = key, float(candidate), metrics
            thresholds[class_id] = best_threshold
    final_metrics = operating_metrics(y_true, probabilities, thresholds)
    return thresholds, final_metrics


def fit_temperature(y_true: np.ndarray, probabilities: np.ndarray) -> float:
    pseudo_logits = np.log(np.clip(probabilities, 1e-9, 1.0))

    def negative_log_likelihood(log_temperature: float) -> float:
        temperature = float(np.exp(log_temperature))
        calibrated = softmax_np(pseudo_logits, temperature)
        return float(-np.log(np.clip(calibrated[np.arange(len(y_true)), y_true], 1e-12, 1.0)).mean())

    result = minimize_scalar(negative_log_likelihood, bounds=(-3.0, 3.0), method="bounded")
    return float(np.exp(result.x)) if result.success else 1.0


def fit_record_temperature(
    records: list[dict[str, Any]],
    pooling: str,
    context_frames: str | int,
) -> float:
    def negative_log_likelihood(log_temperature: float) -> float:
        temperature = float(np.exp(log_temperature))
        y_true, probabilities, _ = records_to_arrays(
            records, pooling, context_frames, temperature
        )
        return float(-np.log(
            np.clip(probabilities[np.arange(len(y_true)), y_true], 1e-12, 1.0)
        ).mean())

    result = minimize_scalar(negative_log_likelihood, bounds=(-3.0, 3.0), method="bounded")
    return float(np.exp(result.x)) if result.success else 1.0



## 6. Inner CV 실행기



In [ ]:
POOLINGS = ["mean_probability", "topk_probability", "logit_logmeanexp"]


def train_fold(
    config: FamilyConfig,
    train_rows: pd.DataFrame,
    val_rows: pd.DataFrame,
    fold_number: int,
) -> tuple[tf.keras.Model, int, dict[str, list[float]]]:
    assert set(train_rows["group_id"]).isdisjoint(set(val_rows["group_id"]))
    x_train, y_train, w_train = build_training_arrays(train_rows, config)
    x_val, y_val, w_val = build_validation_arrays(val_rows, config.representation)
    model = build_classifier(config, x_train, SEED + fold_number, w_train)
    history = model.fit(
        x_train,
        y_train,
        sample_weight=w_train,
        validation_data=(x_val, y_val, w_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=10, restore_best_weights=True, min_delta=1e-4
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5
            ),
        ],
    )
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    return model, best_epoch, history.history


def predict_frame_records(
    model: tf.keras.Model,
    rows: pd.DataFrame,
    representation: str,
    fold_number: int,
) -> list[dict[str, Any]]:
    records = []
    for row in rows.itertuples(index=False):
        embeddings, _ = cached_features(row)
        model_input = embeddings.mean(axis=0, keepdims=True) if representation == "clip_mean" else embeddings
        logits = model.predict(model_input, batch_size=256, verbose=0)
        records.append({
            "relative_path": row.relative_path,
            "group_id": row.group_id,
            "label_id": int(row.label_id),
            "fold": fold_number,
            "frame_logits": logits.astype(np.float32),
        })
    return records


def records_to_arrays(
    records: list[dict[str, Any]],
    pooling: str,
    context_frames: str | int = "full",
    temperature: float = 1.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    ordered = sorted(records, key=lambda item: item["relative_path"])
    y_true = np.asarray([item["label_id"] for item in ordered], dtype=np.int64)
    groups = np.asarray([item["group_id"] for item in ordered], dtype=object)
    probabilities = np.stack([
        aggregate_context(item["frame_logits"], pooling, context_frames, temperature)
        for item in ordered
    ])
    return y_true, probabilities, groups


def summarize_records(
    records: list[dict[str, Any]],
    pooling: str,
    context_frames: str | int = "full",
    temperature: float = 1.0,
    threshold_mode: str = "global",
) -> dict[str, Any]:
    y_true, probabilities, groups = records_to_arrays(records, pooling, context_frames, temperature)
    global_threshold, metrics = tune_global_threshold(y_true, probabilities)
    thresholds: float | np.ndarray = global_threshold
    if threshold_mode == "per_class":
        thresholds, metrics = tune_class_thresholds(y_true, probabilities, global_threshold)
    ordered = sorted(records, key=lambda item: item["relative_path"])
    fold_ids = np.asarray([item["fold"] for item in ordered], dtype=np.int64)
    fold_metrics = [
        operating_metrics(y_true[fold_ids == fold], probabilities[fold_ids == fold], thresholds)
        for fold in np.unique(fold_ids)
    ]
    metrics["cv_target_macro_f1_mean"] = float(np.mean([
        item["target_macro_f1"] for item in fold_metrics
    ]))
    metrics["cv_target_macro_f1_std"] = float(np.std([
        item["target_macro_f1"] for item in fold_metrics
    ]))
    metrics["cv_false_alert_rate_mean"] = float(np.mean([
        item["false_alert_rate"] for item in fold_metrics if np.isfinite(item["false_alert_rate"])
    ]))
    return {
        "pooling": pooling,
        "context_frames": context_frames,
        "temperature": float(temperature),
        "threshold_mode": threshold_mode,
        "thresholds": thresholds,
        "metrics": metrics,
        "y_true": y_true,
        "probabilities": probabilities,
        "groups": groups,
    }


def run_cv_family(config: FamilyConfig) -> dict[str, Any]:
    print(f"\n[CV] {config}")
    started = time.time()
    records: list[dict[str, Any]] = []
    best_epochs: list[int] = []
    histories = []
    for fold_number, (train_idx, val_idx) in enumerate(inner_splits, start=1):
        train_rows = dev_df.iloc[train_idx].reset_index(drop=True)
        val_rows = dev_df.iloc[val_idx].reset_index(drop=True)
        model, best_epoch, history = train_fold(config, train_rows, val_rows, fold_number)
        records.extend(predict_frame_records(model, val_rows, config.representation, fold_number))
        best_epochs.append(best_epoch)
        histories.append(history)
        print(f"  fold {fold_number}/{len(inner_splits)}: best epoch={best_epoch}")
        del model
        tf.keras.backend.clear_session()

    available_poolings = ["mean_probability"] if config.representation == "clip_mean" else POOLINGS
    summaries = {
        pooling: summarize_records(records, pooling)
        for pooling in available_poolings
    }
    elapsed = time.time() - started
    return {
        "config": config,
        "records": records,
        "best_epochs": best_epochs,
        "histories": histories,
        "summaries": summaries,
        "elapsed_seconds": elapsed,
    }


def candidate_key(candidate: dict[str, Any]) -> tuple[float, float, float]:
    metrics = candidate["summary"]["metrics"]
    feasible = metrics["false_alert_rate"] <= MAX_FALSE_ALERT_RATE + 1e-12
    return float(feasible), metrics["cv_target_macro_f1_mean"], metrics["accuracy"]


def candidate_from_family(family_result: dict[str, Any], pooling: str) -> dict[str, Any]:
    return {
        "family_result": family_result,
        "summary": family_result["summaries"][pooling],
    }


experiment_rows: list[dict[str, Any]] = []


def log_candidate(round_number: int, candidate: dict[str, Any], promoted: bool = False) -> None:
    config = candidate["family_result"]["config"]
    summary = candidate["summary"]
    metrics = summary["metrics"]
    experiment_rows.append({
        "round": round_number,
        "family": config.name,
        "head": config.head,
        "representation": config.representation,
        "augmentation": config.augmentation,
        "class_weighting": config.class_weighting,
        "pooling": summary["pooling"],
        "context_frames": summary["context_frames"],
        "temperature": summary["temperature"],
        "threshold_mode": summary["threshold_mode"],
        "family_cv_elapsed_seconds": candidate["family_result"]["elapsed_seconds"],
        **metrics,
        "promoted": promoted,
    })


def improvement_over(challenger: dict[str, Any], incumbent: dict[str, Any]) -> float:
    return (
        challenger["summary"]["metrics"]["cv_target_macro_f1_mean"]
        - incumbent["summary"]["metrics"]["cv_target_macro_f1_mean"]
    )



## 7. 자동 개선 라운드 1–3



In [ ]:
# Round 1: 현재 방식 재현 vs 프레임별 linear/compact head
baseline_family = run_cv_family(FamilyConfig(
    name="current_clip_mean_mlp",
    head="baseline",
    representation="clip_mean",
))
baseline = candidate_from_family(baseline_family, "mean_probability")

round1_families = [
    run_cv_family(FamilyConfig("frame_linear", "linear", "frame")),
    run_cv_family(FamilyConfig("frame_compact", "compact", "frame")),
]
round1_candidates = [
    candidate_from_family(family, pooling)
    for family in round1_families
    for pooling in family["summaries"]
]
best_round1 = max(round1_candidates, key=candidate_key)
round1_promoted = (
    candidate_key(best_round1)[0] == 1.0
    and improvement_over(best_round1, baseline) >= MIN_PROMOTION
)
champion = best_round1 if round1_promoted else baseline
for candidate in [baseline] + round1_candidates:
    log_candidate(1, candidate, promoted=(candidate is champion))

print("\nRound 1 결과")
print("  기준선 CV macro-F1:", baseline["summary"]["metrics"]["cv_target_macro_f1_mean"])
print("  최고 후보 CV macro-F1:", best_round1["summary"]["metrics"]["cv_target_macro_f1_mean"])
print("  승격:", round1_promoted)

rounds_completed = 1
continue_rounds = round1_promoted
round2_promoted = False
round3_promoted = False



In [ ]:
# Round 2: 승격된 frame head에 train-only 경량 증강과 class weighting 비교
round2_candidates: list[dict[str, Any]] = []
if continue_rounds:
    base_config = champion["family_result"]["config"]
    round2_configs = [
        FamilyConfig("clean_balanced", base_config.head, base_config.representation, "none", True),
        FamilyConfig("light_unweighted", base_config.head, base_config.representation, "light", False),
        FamilyConfig("light_balanced", base_config.head, base_config.representation, "light", True),
    ]
    round2_families = [run_cv_family(config) for config in round2_configs]
    round2_candidates = [
        candidate_from_family(family, pooling)
        for family in round2_families
        for pooling in family["summaries"]
    ]
    best_round2 = max(round2_candidates, key=candidate_key)
    round2_promoted = (
        candidate_key(best_round2)[0] == 1.0
        and improvement_over(best_round2, champion) >= MIN_PROMOTION
    )
    previous_champion = champion
    if round2_promoted:
        champion = best_round2
    for candidate in round2_candidates:
        log_candidate(2, candidate, promoted=(candidate is champion and candidate is not previous_champion))
    rounds_completed = 2
    continue_rounds = round2_promoted
    print("\nRound 2 개선폭:", improvement_over(best_round2, previous_champion))
    print("Round 2 승격:", round2_promoted)
else:
    print("\nRound 1에서 0.5%p 개선이 없어 Round 2·3을 중단합니다.")



In [ ]:
# Round 3: context, temperature calibration, 전역/클래스별 threshold 비교
round3_candidates: list[dict[str, Any]] = []
if continue_rounds:
    records = champion["family_result"]["records"]
    pooling = champion["summary"]["pooling"]
    for context_frames in [1, 3, "full"]:
        temperature = fit_record_temperature(records, pooling, context_frames)
        for candidate_temperature in [1.0, temperature]:
            for threshold_mode in ["global", "per_class"]:
                summary = summarize_records(
                    records,
                    pooling,
                    context_frames=context_frames,
                    temperature=candidate_temperature,
                    threshold_mode=threshold_mode,
                )
                round3_candidates.append({
                    "family_result": champion["family_result"],
                    "summary": summary,
                })

    best_round3 = max(round3_candidates, key=candidate_key)
    round3_promoted = (
        candidate_key(best_round3)[0] == 1.0
        and improvement_over(best_round3, champion) >= MIN_PROMOTION
    )
    previous_champion = champion
    if round3_promoted:
        champion = best_round3
    for candidate in round3_candidates:
        log_candidate(3, candidate, promoted=(candidate is champion and candidate is not previous_champion))
    rounds_completed = 3
    print("\nRound 3 개선폭:", improvement_over(best_round3, previous_champion))
    print("Round 3 승격:", round3_promoted)
elif rounds_completed == 2:
    print("\nRound 2에서 0.5%p 개선이 없어 Round 3을 중단합니다.")

experiment_results = pd.DataFrame(experiment_rows)
assert 1 <= rounds_completed <= 3
assert not (rounds_completed >= 2 and not round1_promoted)
assert not (rounds_completed == 3 and not round2_promoted)
experiment_results.to_csv(OUTPUT_DIR / "experiment_results.csv", index=False, encoding="utf-8-sig")
display(experiment_results.sort_values(["round", "cv_target_macro_f1_mean"], ascending=[True, False]))

champion_config: FamilyConfig = champion["family_result"]["config"]
champion_summary = champion["summary"]
print("\n최종 선택 구성")
print("  config:", champion_config)
print("  pooling:", champion_summary["pooling"])
print("  context:", champion_summary["context_frames"])
print("  threshold mode:", champion_summary["threshold_mode"])
print("  development metrics:", champion_summary["metrics"])



## 8. Untouched test 1회 평가



In [ ]:
def train_fixed_epochs(
    rows: pd.DataFrame,
    config: FamilyConfig,
    epochs: int,
    seed: int,
) -> tuple[tf.keras.Model, dict[str, list[float]]]:
    x_train, y_train, sample_weight = build_training_arrays(rows, config)
    model = build_classifier(config, x_train, seed, sample_weight)
    history = model.fit(
        x_train,
        y_train,
        sample_weight=sample_weight,
        epochs=max(1, int(epochs)),
        batch_size=BATCH_SIZE,
        verbose=0,
    )
    return model, history.history


selected_epochs = max(1, int(round(np.median(champion["family_result"]["best_epochs"]))))
test_model, dev_fit_history = train_fixed_epochs(
    dev_df, champion_config, selected_epochs, SEED + 100
)
test_records = predict_frame_records(test_model, test_df, champion_config.representation, fold_number=-1)
test_y, test_probabilities, test_groups = records_to_arrays(
    test_records,
    champion_summary["pooling"],
    champion_summary["context_frames"],
    champion_summary["temperature"],
)
test_thresholds = champion_summary["thresholds"]
test_metrics = operating_metrics(test_y, test_probabilities, test_thresholds)
test_predictions = decision_predictions(test_probabilities, test_thresholds)


def grouped_bootstrap_interval(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    groups: np.ndarray,
    thresholds: float | np.ndarray,
    iterations: int = BOOTSTRAP_ITERATIONS,
) -> dict[str, list[float]]:
    unique_groups = np.unique(groups)
    group_indices = {group: np.flatnonzero(groups == group) for group in unique_groups}
    rng = np.random.default_rng(SEED + 999)
    values = {"target_macro_f1": [], "false_alert_rate": [], "accuracy": []}
    for _ in range(iterations):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        indices = np.concatenate([group_indices[group] for group in sampled_groups])
        metrics = operating_metrics(y_true[indices], probabilities[indices], thresholds)
        for key in values:
            if np.isfinite(metrics[key]):
                values[key].append(metrics[key])
    return {
        key: [float(np.percentile(series, 2.5)), float(np.percentile(series, 97.5))]
        for key, series in values.items() if series
    }


test_confidence_intervals = grouped_bootstrap_interval(
    test_y, test_probabilities, test_groups, test_thresholds
)

# 동일한 untouched test 호출에서 사전 선언된 기존 구조 기준선도 함께 평가합니다.
if champion_config == baseline["family_result"]["config"]:
    baseline_test_metrics = dict(test_metrics)
else:
    baseline_epochs = max(1, int(round(np.median(baseline["family_result"]["best_epochs"]))))
    baseline_test_model, _ = train_fixed_epochs(
        dev_df, baseline["family_result"]["config"], baseline_epochs, SEED + 101
    )
    baseline_test_records = predict_frame_records(
        baseline_test_model,
        test_df,
        baseline["family_result"]["config"].representation,
        fold_number=-4,
    )
    baseline_test_y, baseline_test_probabilities, _ = records_to_arrays(
        baseline_test_records,
        baseline["summary"]["pooling"],
        baseline["summary"]["context_frames"],
        baseline["summary"]["temperature"],
    )
    baseline_test_metrics = operating_metrics(
        baseline_test_y,
        baseline_test_probabilities,
        baseline["summary"]["thresholds"],
    )
    del baseline_test_model
    tf.keras.backend.clear_session()

print("Test metrics:", test_metrics)
print("Baseline test metrics:", baseline_test_metrics)
print("95% grouped bootstrap CI:", test_confidence_intervals)
print(classification_report(
    test_y,
    test_predictions,
    labels=range(NUM_CLASSES),
    target_names=CATEGORIES,
    zero_division=0,
))



## 9. 전체 데이터 재학습 및 TFLite 내보내기



In [ ]:
final_model, final_history = train_fixed_epochs(
    metadata, champion_config, selected_epochs, SEED + 200
)

temperature_layer = tf.keras.layers.Rescaling(
    scale=1.0 / max(float(champion_summary["temperature"]), 1e-4),
    name="temperature_scaling",
)(final_model.output)
probability_output = tf.keras.layers.Softmax(name="probabilities")(temperature_layer)
deployment_model = tf.keras.Model(
    final_model.input,
    probability_output,
    name="hearo_classifier_v2",
)
deployment_model.save(OUTPUT_DIR / "hearo_classifier_v2.keras")


def convert_tflite(model: tf.keras.Model, dynamic_range: bool) -> bytes:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    if dynamic_range:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    return converter.convert()


float_tflite = convert_tflite(deployment_model, dynamic_range=False)
dynamic_tflite = convert_tflite(deployment_model, dynamic_range=True)
(OUTPUT_DIR / "hearo_classifier_v2_float32.tflite").write_bytes(float_tflite)
(OUTPUT_DIR / "hearo_classifier_v2_dynamic.tflite").write_bytes(dynamic_tflite)


def tflite_predict(interpreter: tf.lite.Interpreter, embeddings: np.ndarray) -> np.ndarray:
    input_details = interpreter.get_input_details()[0]
    requested_shape = [len(embeddings), 1024]
    if list(input_details["shape"]) != requested_shape:
        interpreter.resize_tensor_input(input_details["index"], requested_shape, strict=False)
        interpreter.allocate_tensors()
        input_details = interpreter.get_input_details()[0]
    interpreter.set_tensor(input_details["index"], embeddings.astype(input_details["dtype"]))
    interpreter.invoke()
    output_details = interpreter.get_output_details()[0]
    output = interpreter.get_tensor(output_details["index"])
    if np.issubdtype(output_details["dtype"], np.integer):
        scale, zero_point = output_details["quantization"]
        output = (output.astype(np.float32) - zero_point) * scale
    return np.asarray(output, np.float32)


def tflite_records(model_content: bytes, rows: pd.DataFrame) -> list[dict[str, Any]]:
    interpreter = tf.lite.Interpreter(model_content=model_content)
    interpreter.allocate_tensors()
    records = []
    for row in rows.itertuples(index=False):
        embeddings, _ = cached_features(row)
        model_input = embeddings.mean(axis=0, keepdims=True) if champion_config.representation == "clip_mean" else embeddings
        probabilities = np.clip(tflite_predict(interpreter, model_input), 1e-9, 1.0)
        records.append({
            "relative_path": row.relative_path,
            "group_id": row.group_id,
            "label_id": int(row.label_id),
            "fold": -2,
            # aggregate_context는 logits 입력이므로 log(probability)를 사용합니다.
            "frame_logits": np.log(probabilities),
        })
    return records


def evaluate_tflite(model_content: bytes) -> tuple[dict[str, float], np.ndarray]:
    records = tflite_records(model_content, test_df)
    y_true, probabilities, _ = records_to_arrays(
        records,
        champion_summary["pooling"],
        champion_summary["context_frames"],
        temperature=1.0,
    )
    return operating_metrics(y_true, probabilities, test_thresholds), probabilities


float_metrics, float_probabilities = evaluate_tflite(float_tflite)
dynamic_metrics, dynamic_probabilities = evaluate_tflite(dynamic_tflite)
quantized_drop = float_metrics["target_macro_f1"] - dynamic_metrics["target_macro_f1"]

keras_final_records = []
for row in test_df.itertuples(index=False):
    embeddings, _ = cached_features(row)
    model_input = embeddings.mean(axis=0, keepdims=True) if champion_config.representation == "clip_mean" else embeddings
    probabilities = deployment_model.predict(model_input, verbose=0)
    keras_final_records.append({
        "relative_path": row.relative_path,
        "group_id": row.group_id,
        "label_id": int(row.label_id),
        "fold": -3,
        "frame_logits": np.log(np.clip(probabilities, 1e-9, 1.0)),
    })
_, keras_final_probabilities, _ = records_to_arrays(
    keras_final_records,
    champion_summary["pooling"],
    champion_summary["context_frames"],
    temperature=1.0,
)
float_max_abs_error = float(np.max(np.abs(keras_final_probabilities - float_probabilities)))
dynamic_max_abs_error = float(np.max(np.abs(keras_final_probabilities - dynamic_probabilities)))
if float_max_abs_error > 1e-4:
    raise RuntimeError(
        f"Keras/float32 TFLite 확률 오차가 허용값을 초과했습니다: {float_max_abs_error:.6f}"
    )
use_dynamic = quantized_drop <= 0.005 and dynamic_max_abs_error <= 0.02
selected_tflite = dynamic_tflite if use_dynamic else float_tflite
(OUTPUT_DIR / "hearo_classifier_v2.tflite").write_bytes(selected_tflite)
selected_probabilities = dynamic_probabilities if use_dynamic else float_probabilities
tflite_max_abs_error = float(np.max(np.abs(keras_final_probabilities - selected_probabilities)))


def benchmark_tflite(model_content: bytes, example: np.ndarray, repeats: int = 100) -> float:
    interpreter = tf.lite.Interpreter(model_content=model_content)
    interpreter.allocate_tensors()
    for _ in range(5):
        tflite_predict(interpreter, example)
    started = time.perf_counter()
    for _ in range(repeats):
        tflite_predict(interpreter, example)
    return float((time.perf_counter() - started) * 1_000.0 / repeats)


benchmark_row = next(test_df.itertuples(index=False))
benchmark_embeddings, _ = cached_features(benchmark_row)
benchmark_input = (
    benchmark_embeddings.mean(axis=0, keepdims=True)
    if champion_config.representation == "clip_mean"
    else benchmark_embeddings
)
float_inference_ms = benchmark_tflite(float_tflite, benchmark_input)
dynamic_inference_ms = benchmark_tflite(dynamic_tflite, benchmark_input)

threshold_values = (
    np.full(NUM_CLASSES, float(test_thresholds))
    if np.isscalar(test_thresholds)
    else np.asarray(test_thresholds, dtype=float)
)
threshold_values[UNKNOWN_ID] = 0.0

with (OUTPUT_DIR / "categories_v2.txt").open("w", encoding="utf-8") as file:
    file.write("\n".join(CATEGORIES) + "\n")

model_metadata = {
    "schema_version": 2,
    "model_name": "hearo_classifier_v2",
    "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    "sample_rate": TARGET_SR,
    "record_step_seconds": 1.0,
    "rolling_buffer_seconds": 2.0,
    "yamnet_handle": YAMNET_HANDLE,
    "yamnet_frame_seconds": YAMNET_FRAME_SECONDS,
    "yamnet_hop_seconds": 0.48,
    "classifier_input": {"dtype": "float32", "shape": [None, 1024]},
    "classifier_output": {"dtype": "float32", "shape": [None, NUM_CLASSES], "type": "probabilities"},
    "categories": CATEGORIES,
    "unknown_label": UNKNOWN_LABEL,
    "representation": champion_config.representation,
    "frame_pooling": champion_summary["pooling"],
    "context_frames": champion_summary["context_frames"],
    "temperature": float(champion_summary["temperature"]),
    "temperature_embedded_in_tflite": True,
    "threshold_mode": champion_summary["threshold_mode"],
    "class_thresholds": {
        label: float(threshold_values[index]) for index, label in enumerate(CATEGORIES)
    },
    "unknown_false_alert_limit": MAX_FALSE_ALERT_RATE,
    "yamnet_gate": {"enabled": False, "reason": "비표적음 class가 2차 분류에서 거부를 담당"},
    "selected_tflite_variant": "dynamic_range" if use_dynamic else "float32",
    "selected_epochs": selected_epochs,
    "rounds_completed": rounds_completed,
    "class_mapping": {
        "노크_목재": "노크소리",
        "노크_철재문": "노크소리",
        "도어락_개방음": "도어락소리",
        "도어락_입력음": "도어락소리",
        "사이렌_삐뽀삐뽀": "비상벨소리",
        "사이렌_안내음": "비상벨소리",
        "사이렌_애애애애앵": "비상벨소리",
        "사이렌_철철철": "비상벨소리",
        "아기 울음": "아기울음소리",
    },
}
(OUTPUT_DIR / "model_metadata_v2.json").write_text(
    json.dumps(model_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)



## 10. 결과 리포트와 시각화



In [ ]:
def json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, FamilyConfig):
        return asdict(value)
    return value


metrics_report = {
    "selection_data": "development grouped out-of-fold",
    "baseline_development": baseline["summary"]["metrics"],
    "baseline_untouched_test": baseline_test_metrics,
    "champion_development": champion_summary["metrics"],
    "untouched_test": test_metrics,
    "untouched_test_group_bootstrap_95ci": test_confidence_intervals,
    "champion_config": champion_config,
    "champion_decision": {
        "pooling": champion_summary["pooling"],
        "context_frames": champion_summary["context_frames"],
        "temperature": champion_summary["temperature"],
        "threshold_mode": champion_summary["threshold_mode"],
        "thresholds": test_thresholds,
    },
    "rounds_completed": rounds_completed,
    "promotion_delta": MIN_PROMOTION,
    "false_alert_limit": MAX_FALSE_ALERT_RATE,
    "tflite": {
        "float32_metrics_on_parity_set": float_metrics,
        "dynamic_metrics_on_parity_set": dynamic_metrics,
        "selected": "dynamic_range" if use_dynamic else "float32",
        "float32_bytes": len(float_tflite),
        "dynamic_bytes": len(dynamic_tflite),
        "selected_bytes": len(selected_tflite),
        "keras_tflite_max_abs_probability_error": tflite_max_abs_error,
        "float32_max_abs_probability_error": float_max_abs_error,
        "dynamic_max_abs_probability_error": dynamic_max_abs_error,
        "float32_colab_inference_ms": float_inference_ms,
        "dynamic_colab_inference_ms": dynamic_inference_ms,
        "benchmark_frames": len(benchmark_input),
    },
    "environment": {
        "tensorflow": tf.__version__,
        "tensorflow_hub": getattr(hub, "__version__", "unknown"),
        "python": os.sys.version,
    },
}
(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps(json_ready(metrics_report), ensure_ascii=False, indent=2), encoding="utf-8"
)

# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(dev_fit_history["loss"], label="Development fit")
axes[0].set_title("Selected model loss before untouched test")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].plot(final_history["loss"], label="All-data deployment fit", color="#ED7D31")
axes[1].set_title("Deployment model loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "1_learning_curves.png", dpi=200, bbox_inches="tight")
plt.show()

# Untouched test confusion matrix
cm = confusion_matrix(test_y, test_predictions, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIES, yticklabels=CATEGORIES, ax=ax)
ax.set_title("Untouched grouped test confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "2_test_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

# 클래스별 test 지표
precision, recall, class_f1, support = precision_recall_fscore_support(
    test_y, test_predictions, labels=range(NUM_CLASSES), zero_division=0
)
per_class = pd.DataFrame({
    "class": CATEGORIES,
    "precision": precision,
    "recall": recall,
    "f1": class_f1,
    "support": support,
})
per_class.to_csv(OUTPUT_DIR / "test_per_class_metrics.csv", index=False, encoding="utf-8-sig")
per_class.set_index("class")[["precision", "recall", "f1"]].plot.bar(figsize=(15, 6))
plt.ylim(0, 1.05)
plt.title("Untouched grouped test per-class metrics")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "3_test_per_class_metrics.png", dpi=200, bbox_inches="tight")
plt.show()

# Development OOF threshold trade-off
dev_y = champion_summary["y_true"]
dev_probabilities = champion_summary["probabilities"]
threshold_curve = []
for threshold in np.linspace(0.0, 1.0, 101):
    metrics = operating_metrics(dev_y, dev_probabilities, threshold)
    threshold_curve.append({"threshold": threshold, **metrics})
threshold_curve = pd.DataFrame(threshold_curve)
threshold_curve.to_csv(OUTPUT_DIR / "threshold_tradeoff.csv", index=False, encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(threshold_curve["threshold"], threshold_curve["target_macro_f1"], label="Target macro-F1")
ax.plot(threshold_curve["threshold"], threshold_curve["false_alert_rate"], label="Unknown false-alert rate")
ax.axhline(MAX_FALSE_ALERT_RATE, color="red", linestyle="--", label="False-alert limit")
ax.set_xlabel("Global threshold")
ax.set_ylabel("Score / rate")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "4_threshold_tradeoff.png", dpi=200, bbox_inches="tight")
plt.show()

print("\n=== 완료 ===")
print(f"기준선 development CV macro-F1: {baseline['summary']['metrics']['cv_target_macro_f1_mean']:.4f}")
print(f"선택 모델 development CV macro-F1: {champion_summary['metrics']['cv_target_macro_f1_mean']:.4f}")
print(f"Untouched test macro-F1: {test_metrics['target_macro_f1']:.4f}")
print(f"Untouched test false-alert rate: {test_metrics['false_alert_rate']:.4f}")
print(f"TFLite 선택: {'dynamic_range' if use_dynamic else 'float32'} ({len(selected_tflite)/1024:.1f} KiB)")
print("산출물:", OUTPUT_DIR)
